# Deep Learning 084 — The Transformer at Inference

Companion notebook to the lesson, and the last one on the architecture. The encoder is
unchanged from training; the decoder becomes **auto-regressive**. This notebook builds the
generation loop and then tests the two claims that most descriptions get slightly wrong.

| Step | What we measure |
|---|---|
| only the last vector | equivalent to projecting all and keeping the last |
| the waste | 8-word sentence: **36 vectors computed, 8 used** — 77.8% discarded |
| **does masking matter at inference?** | 1 block: **exactly 0.0000**. 6 blocks: **0.3523** |
| KV caching, correctness | logits agree to **1.1e-06** — exact, not approximate |
| KV caching, speed | only **a few ×** measured, against **8.5×–64.5×** predicted |

Needs `torch` (CPU is fine). No training.

In [ ]:
import math, time
import torch
import torch.nn as nn

D, D_FF, H = 256, 1024, 8

def causal_mask(n):
    return torch.full((n, n), float("-inf")).triu(1)

class MHA(nn.Module):
    '''One class for both flavours: pass the same tensor twice for self-attention,
       or (decoder, encoder) for cross attention.'''
    def __init__(self, d=D, h=H):
        super().__init__()
        self.h, self.dk = h, d // h
        self.wq, self.wk = nn.Linear(d, d), nn.Linear(d, d)
        self.wv, self.wo = nn.Linear(d, d), nn.Linear(d, d)

    def forward(self, q_src, kv_src, masked=False):
        nq, nk = len(q_src), len(kv_src)
        q = self.wq(q_src).view(nq, self.h, self.dk).transpose(0, 1)
        k = self.wk(kv_src).view(nk, self.h, self.dk).transpose(0, 1)
        v = self.wv(kv_src).view(nk, self.h, self.dk).transpose(0, 1)
        s = q @ k.transpose(-2, -1) / math.sqrt(self.dk)
        if masked:
            s = s + causal_mask(nq)
        w = torch.softmax(s, dim=-1)
        return self.wo((w @ v).transpose(0, 1).reshape(nq, self.h * self.dk))

class DecoderBlock(nn.Module):
    '''THREE sub-blocks, not two.'''
    def __init__(self, d=D, d_ff=D_FF):
        super().__init__()
        self.self_attn, self.cross_attn = MHA(d), MHA(d)
        self.ff = nn.Sequential(nn.Linear(d, d_ff), nn.ReLU(), nn.Linear(d_ff, d))
        self.ln1, self.ln2, self.ln3 = nn.LayerNorm(d), nn.LayerNorm(d), nn.LayerNorm(d)

    def forward(self, x, enc, masked=True):
        x = self.ln1(x + self.self_attn(x, x, masked=masked))   # lesson 081
        x = self.ln2(x + self.cross_attn(x, enc))               # lesson 082
        x = self.ln3(x + self.ff(x))                            # lesson 080
        return x

## Part A — The generation loop

Two rules define it: the sequence grows by one token per step, and **only the last vector is
projected**.

In [ ]:
def make_decoder(n_blocks, seed):
    torch.manual_seed(seed)
    return nn.ModuleList([DecoderBlock() for _ in range(n_blocks)])

torch.manual_seed(1084)
blocks = make_decoder(6, 11)
torch.manual_seed(1084)
head = nn.Linear(D, 2000)
enc = torch.randn(3, D)            # "We are friends", already encoded
x = torch.randn(5, D)              # five tokens generated so far

with torch.no_grad():
    y = x
    for b in blocks:
        y = b(y, enc)
    all_logits, last_logits = head(y), head(y[-1:])

print("project all 5 rows, keep the last :", tuple(all_logits[-1:].shape))
print("project only the last row         :", tuple(last_logits.shape))
print(f"max difference                    : {(all_logits[-1:] - last_logits).abs().max():.2e}")

The earlier vectors are not wrong — they are *last step's* answers, already emitted.
Projecting them again would repeat words forever. So each step computes $t$ vectors and
throws away $t-1$.

In [ ]:
print(f"{'step':>6} {'tokens fed':>12} {'vectors made':>14} {'used':>6} {'discarded':>11}")
made = used = 0
for t in range(1, 9):
    made += t; used += 1
    print(f"{t:>6} {t:>12} {t:>14} {1:>6} {t-1:>11}")
print(f"\nover 8 words: {made} computed, {used} used, {100*(made-used)/made:.1f}% discarded")

## Part B — Does masking matter at inference?

The usual answer is "yes, masking is not training-only". That is correct, and the reasoning
normally offered for it is not.

Think about what is actually used: the emitted word comes from the **last** row of the
attention matrix, and that row attends to positions $0 \ldots t-1$ — all in the past. **The
causal mask blocks nothing in that row.** So for a one-block decoder, masking cannot change
the emitted token at all.

Predict the four numbers before running this.

In [ ]:
print(f"{'blocks':>8} {'max |masked - unmasked| at the LAST position':>46}")
for n_blocks in (1, 2, 3, 6):
    torch.manual_seed(2084)
    blocks_ = make_decoder(n_blocks, 22)
    enc_ = torch.randn(3, D)
    x_ = torch.randn(7, D)
    with torch.no_grad():
        ym = yu = x_
        for b in blocks_: ym = b(ym, enc_, masked=True)
        for b in blocks_: yu = b(yu, enc_, masked=False)
    diff = (ym[-1] - yu[-1]).abs().max().item()
    note = "  <- EXACTLY zero" if diff == 0.0 else ""
    print(f"{n_blocks:>8} {diff:>46.4f}{note}")

**One block: exactly zero. Six blocks: a completely different vector.**

The route is indirect. Block 2's last position reads block 1's outputs at **every** position,
and those earlier outputs *are* the ones the mask changes. So the mask reaches the emitted
token through **depth**, not because the last row needed masking.

The practical rule survives intact and is the part worth keeping: **whatever you did to the
training input, do to the query input.** A model trained on masked positions and served
without them is being fed a distribution it never saw — a general machine-learning principle,
not something specific to transformers.

## Part C — KV caching

At step $t$ the naive loop re-derives keys and values for all $t$ tokens. But $t-1$ of them
were computed at step $t-1$ and **cannot** have changed: causal masking guarantees token $j$'s
representation never depends on anything after $j$. That is lesson 081's prefix consistency
showing up as an engineering opportunity.

In [ ]:
def naive(blocks, head, enc, start, n_steps):
    seq, out = start.clone(), []
    with torch.no_grad():
        for _ in range(n_steps):
            y = seq
            for b in blocks:
                y = b(y, enc, masked=True)
            out.append(head(y[-1]))
            seq = torch.cat([seq, y[-1:]], dim=0)
    return torch.stack(out)

def cached(blocks, head, enc, start, n_steps):
    seq, out = start.clone(), []
    enc_kv = []
    for b in blocks:                       # the encoder does not change: project K/V ONCE
        n = len(enc)
        k = b.cross_attn.wk(enc).view(n, H, -1).transpose(0, 1)
        v = b.cross_attn.wv(enc).view(n, H, -1).transpose(0, 1)
        enc_kv.append((k, v))
    self_kv = [None] * len(blocks)
    with torch.no_grad():
        for step in range(n_steps):
            y = seq if step == 0 else seq[-1:]
            for i, b in enumerate(blocks):
                n = len(y)
                k = b.self_attn.wk(y).view(n, H, -1).transpose(0, 1)
                v = b.self_attn.wv(y).view(n, H, -1).transpose(0, 1)
                self_kv[i] = (k, v) if self_kv[i] is None else \
                    (torch.cat([self_kv[i][0], k], 1), torch.cat([self_kv[i][1], v], 1))
                if step == 0:
                    a = b.self_attn(y, y, masked=True)
                else:
                    kk, vv = self_kv[i]
                    q = b.self_attn.wq(y).view(n, H, -1).transpose(0, 1)
                    w = torch.softmax(q @ kk.transpose(-2, -1) / math.sqrt(b.self_attn.dk), -1)
                    a = b.self_attn.wo((w @ vv).transpose(0, 1).reshape(n, D))
                h = b.ln1(y + a)
                kk, vv = enc_kv[i]
                q = b.cross_attn.wq(h).view(len(h), H, -1).transpose(0, 1)
                w = torch.softmax(q @ kk.transpose(-2, -1) / math.sqrt(b.cross_attn.dk), -1)
                cx = b.cross_attn.wo((w @ vv).transpose(0, 1).reshape(len(h), D))
                h = b.ln2(h + cx)
                y = b.ln3(h + b.ff(h))
            out.append(head(y[-1]))
            seq = torch.cat([seq, y[-1:]], dim=0)
    return torch.stack(out)

torch.manual_seed(4084); blocks = make_decoder(6, 44)
torch.manual_seed(4084); head = nn.Linear(D, 2000)
enc, start = torch.randn(8, D), torch.randn(1, D)

a, b_ = naive(blocks, head, enc, start, 24), cached(blocks, head, enc, start, 24)
print(f"max |naive logits - cached logits| over 24 steps : {(a-b_).abs().max():.2e}")

**The cache is an optimisation, not an approximation.** Now the part that does not go the way
the arithmetic suggests.

In [ ]:
def best_of(fn, n_steps, repeats=3):
    best = float("inf")
    for _ in range(repeats):
        t0 = time.perf_counter(); fn(blocks, head, enc, start, n_steps)
        best = min(best, time.perf_counter() - t0)
    return best

print(f"{'steps':>7} {'naive':>10} {'cached':>10} {'measured':>10} {'predicted':>11}")
for n_steps in (16, 32, 64):
    tn, tc = best_of(naive, n_steps), best_of(cached, n_steps)
    print(f"{n_steps:>7} {tn:>9.4f}s {tc:>9.4f}s {tn/tc:>9.1f}x {(n_steps+1)/2:>10.1f}x")

**A result that falls short of its own arithmetic.** The cache removes work growing as $n^2$,
so the ratio should be about $(n+1)/2$. Measured, it is a few times — far less, growing far
more slowly.

The reason: at this size (6 blocks, $d = 256$, CPU) run time is dominated by **per-call
overhead** — Python dispatch and small kernel launches — which the cache does not reduce,
since both versions make the same number of block calls per step. The asymptotic argument is
sound; it simply does not govern here. That the speedup *does* arrive for production models,
where matrices are large enough for arithmetic to dominate and KV-cache **memory** becomes the
limit on batch size, is **reported, not measured** — that regime is not reachable on a laptop.

In [ ]:
# what the cache saves, in attention rows -- this part is arithmetic, not timing
print(f"{'n':>8} {'naive rows':>14} {'cached rows':>13} {'ratio':>9}")
for n in (8, 32, 128, 512):
    print(f"{n:>8} {n*(n+1)//2:>14,} {n:>13,} {n*(n+1)//2/n:>8.1f}x")

One caveat worth stating precisely, because it is widely misremembered: **the cache does not
make attention itself cheaper.** Each new token still attends over every previous token, so
total attention work stays $O(n^2)$. What the cache removes is the *re-derivation* of keys and
values for tokens whose representations provably cannot have changed.

## What to take away

- **The encoder is identical** at training and inference; only the decoder changes.
- **Auto-regressive:** SOS in, one word out, growing by one token per step until EOS.
- **Only the last vector is projected** — 77.8% of an 8-word generation is discarded.
- **Masking applies at inference**, but changes the emitted vector by **exactly 0.0000** at
  depth 1 and **0.3523** at depth 6. It reaches the output through depth.
- **KV caching is exact** (1.1e-06) but its speedup is overhead-bound at this scale.
- **Greedy decoding is a choice.** Beam search, temperature, top-$k$ and nucleus sampling
  change only how the final distribution is read.

**Exercises**

1. Add an EOS token and a real stopping condition to the loop in Part C.
2. Replace `argmax` with temperature sampling. At what temperature does the output stop being
   repetitive, and what breaks at very high temperature?
3. In Part B, try depths 12 and 24. Does the mask's effect keep growing, and does it level
   off?
4. Time `cached` with `d = 1024` and 12 blocks. Does the measured speedup move towards the
   predicted one? (This is the experiment the lesson could not run.)